# 02 — Training & Evaluasi Isolation Forest (Versi 2)

**Pertanyaan:** apakah fitur agregasi jendela waktu menaikkan deteksi port scan dan brute force SSH,
**tanpa** menaikkan alarm palsu?

Tiga rancangan dibandingkan. Semuanya hanya memakai **Isolation Forest**, sesuai batasan masalah.

| Rancangan | Isi |
|---|---|
| **A** | 1 model, 68 fitur aliran (baseline) |
| **B** | 1 model, 68 fitur aliran + 7 fitur agregasi digabung jadi 75 fitur |
| **A+C** | 2 model: **A** (68 fitur aliran) dan **C** (7 fitur agregasi saja = *perilaku sumber*). Aliran ditandai anomali bila A **atau** C menandainya |

**Aturan yang sama untuk semua rancangan:**
- Dilatih **hanya** dengan lalu lintas normal hari Senin.
- `n_estimators=100`, `max_samples=256`, `random_state=42`; StandardScaler di-fit pada data latih.
- Ambang tiap model = kuantil skor data latih pada (1 − contamination).
- Contamination dan panjang jendela dipilih di data **validasi**: F1 tertinggi dengan syarat **FPR validasi ≤ 10%**.
  Batas FPR dipakai karena pada sistem pemantauan, alarm palsu yang terlalu banyak membuat admin mengabaikan notifikasi.
- Data **uji** hanya dipakai **sekali**, di akhir.

In [ ]:
import os, sys

# Lokasi kerja: Google Colab (Drive) atau laptop
try:
    from google.colab import drive
    drive.mount('/content/drive')
    FOLDER = '/content/drive/MyDrive/TA/CIC-IDS2017'          # folder notebook ini di Google Drive
    CALON_KODE = [os.path.join(FOLDER, '2_sistem_nids'),      # lokasi folder kode sistem NIDS yang dicari
                  '/content/drive/MyDrive/TA/2_sistem_nids',
                  '/content/drive/MyDrive/TA/nids-deteksi',
                  '/content/drive/MyDrive/TA/04_Kode/nids-deteksi']
except ImportError:
    FOLDER = os.path.abspath(os.path.join('..', '..', '05_Data_Eksperimen', 'CIC-IDS2017'))  # folder data (notebook ada di 04_Kode/4_notebook)
    CALON_KODE = [os.path.abspath(os.path.join('..', '2_sistem_nids'))]                       # kode sistem NIDS

KODE = next((p for p in CALON_KODE if os.path.isdir(os.path.join(p, 'nids'))), None)
os.makedirs(FOLDER, exist_ok=True)
os.chdir(FOLDER)
DATA = os.path.join(FOLDER, 'data')
print('Folder kerja :', FOLDER)
print('Kode nids    :', KODE or 'tidak ditemukan (hanya dibutuhkan notebook 01)')

In [ ]:
import itertools, json, time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

PROC = os.path.join(DATA, 'processed_v2')
HASIL = os.path.join(FOLDER, 'hasil_v2')
os.makedirs(HASIL, exist_ok=True)
SEED = 42
PARAM = dict(n_estimators=100, max_samples=256, random_state=SEED, n_jobs=-1)
GRID_C = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.30]
BATAS_FPR = 0.10
NAMA_SKENARIO = {'http_dos': 'HTTP flood / DoS', 'port_scan': 'Port scanning',
                 'ssh_bruteforce': 'SSH brute force', 'lainnya': 'Di luar cakupan TA'}

info = json.load(open(os.path.join(PROC, 'fitur_v2.json'), encoding='utf-8'))
F68, AGG, JENDELA = info['fitur_aliran'], info['fitur_agregasi'], info['jendela_detik']
train = pd.read_parquet(os.path.join(PROC, 'train_senin_normal.parquet'))
val = pd.read_parquet(os.path.join(PROC, 'val.parquet'))
test = pd.read_parquet(os.path.join(PROC, 'test.parquet'))
y_val, y_test = val['Label_biner'].to_numpy(), test['Label_biner'].to_numpy()
print('latih %s | validasi %s | uji %s' % (f'{len(train):,}', f'{len(val):,}', f'{len(test):,}'))

## Fungsi bantu

- `metrik` menghitung precision, recall, F1, FPR, dan accuracy dari prediksi.
- `Model` = satu Isolation Forest beserta scaler-nya. Skor dibalik tandanya, jadi **skor tinggi = lebih anomali**.
- `pilih` mencoba semua kombinasi contamination (satu per model), lalu memilih F1 validasi tertinggi dengan FPR ≤ 10%.

In [ ]:
def metrik(y, pred):
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r) if p + r else 0.0,
            'fpr': fp / (fp + tn) if fp + tn else 0.0, 'accuracy': (tp + tn) / len(y),
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}


class Model:
    def __init__(self, nama, fitur):
        self.nama, self.fitur = nama, fitur
        t = time.time()
        self.scaler = StandardScaler().fit(train[fitur])
        self.model = IsolationForest(**PARAM).fit(self.scaler.transform(train[fitur]))
        self.waktu_latih = time.time() - t
        self.s_train = self.skor(train)
        self.s_val = self.skor(val)

    def skor(self, d):
        return -self.model.score_samples(self.scaler.transform(d[self.fitur]))

    def ambang(self, c):
        return float(np.quantile(self.s_train, 1 - c))


def pilih(models):
    baris = []
    for cs in itertools.product(GRID_C, repeat=len(models)):
        pred = np.zeros(len(y_val), bool)
        for m, c in zip(models, cs):
            pred |= m.s_val > m.ambang(c)
        baris.append({**{'c_' + m.nama: c for m, c in zip(models, cs)}, **metrik(y_val, pred)})
    tab = pd.DataFrame(baris)
    ok = tab[tab['fpr'] <= BATAS_FPR]
    best = ok.loc[ok['f1'].idxmax()]
    return {m.nama: float(best['c_' + m.nama]) for m in models}, float(best['f1']), tab

## Melatih semua model
Model A satu buah; model B dan C masing-masing untuk jendela 60, 120, dan 300 detik.

In [ ]:
mA = Model('A', F68)
mB = {w: Model('B', F68 + ['%s_w%d' % (c, w) for c in AGG]) for w in JENDELA}
mC = {w: Model('C', ['%s_w%d' % (c, w) for c in AGG]) for w in JENDELA}
print('waktu latih A: %.1f detik' % mA.waktu_latih)

## Pemilihan contamination & panjang jendela (data validasi)

In [ ]:
cA, f1A, tabA = pilih([mA])
pil_B, pil_AC, rows_w = {}, {}, []
for w in JENDELA:
    cB, f1B, _ = pilih([mB[w]])
    cAC, f1AC, tabAC = pilih([mA, mC[w]])
    pil_B[w], pil_AC[w] = (cB, f1B), (cAC, f1AC, tabAC)
    rows_w.append({'jendela (detik)': w, 'F1 validasi B': f1B, 'F1 validasi A+C': f1AC,
                   'contamination A': cAC['A'], 'contamination C': cAC['C']})
wB = max(JENDELA, key=lambda w: pil_B[w][1])
wC = max(JENDELA, key=lambda w: pil_AC[w][1])
tab_jendela = pd.DataFrame(rows_w)
tab_jendela.to_csv(os.path.join(HASIL, 'tabel_pemilihan_jendela.csv'), index=False)
tabA.to_csv(os.path.join(HASIL, 'tabel_penalaan_A.csv'), index=False)
pil_AC[wC][2].to_csv(os.path.join(HASIL, 'tabel_penalaan_A+C.csv'), index=False)
print('A   : contamination %s, F1 validasi %.4f' % (cA['A'], f1A))
print('Jendela terpilih: B = %d detik, A+C = %d detik' % (wB, wC))
tab_jendela.round(4)

## Evaluasi akhir pada data uji (dijalankan sekali)

ROC-AUC rancangan A+C dihitung dari skor gabungan: skor tiap model dibagi ambangnya, lalu diambil yang terbesar.

In [ ]:
rancangan = {'A': ([mA], cA), 'B': ([mB[wB]], pil_B[wB][0]), 'A+C': ([mA, mC[wC]], pil_AC[wC][0])}
judul = {'A': 'A: 68 fitur aliran', 'B': 'B: 75 fitur dalam 1 model (%d s)' % wB,
         'A+C': 'A+C: 2 model, aliran + perilaku sumber (%d s)' % wC}
hasil, prediksi = {}, {}
for nama, (models, cs) in rancangan.items():
    t = time.time()
    skor = [m.skor(test) for m in models]
    per_aliran_us = (time.time() - t) / len(test) * 1e6
    pred = np.zeros(len(test), bool)
    for m, s in zip(models, skor):
        pred |= s > m.ambang(cs[m.nama])
    s_gab = np.max([s / m.ambang(cs[m.nama]) for m, s in zip(models, skor)], axis=0)
    mt = metrik(y_test, pred)
    mt['roc_auc'] = float(roc_auc_score(y_test, s_gab))
    hasil[nama] = {'metrik': mt, 'contamination': cs, 'fitur': {m.nama: len(m.fitur) for m in models},
                   'waktu_latih_detik': round(sum(m.waktu_latih for m in models), 3),
                   'per_aliran_mikrodetik': round(per_aliran_us, 3)}
    prediksi[nama] = pred

KOL = ['precision', 'recall', 'f1', 'fpr', 'accuracy', 'roc_auc']
banding = pd.DataFrame({judul[n]: [hasil[n]['metrik'][k] for k in KOL] for n in rancangan},
                       index=['Precision', 'Recall', 'F1-score', 'FPR', 'Accuracy', 'ROC-AUC'])
banding.to_csv(os.path.join(HASIL, 'tabel_perbandingan_metrik.csv'))
banding.round(4)

### Detection rate per skenario serangan (data uji)

In [ ]:
rows_s, rows_l = [], []
sk_arr, lab_arr = test['Skenario'].to_numpy(), test['Label'].to_numpy()
for nama, pred in prediksi.items():
    for sk in pd.unique(sk_arr):
        if sk != 'normal':
            msk = sk_arr == sk
            rows_s.append({'rancangan': nama, 'skenario': NAMA_SKENARIO.get(sk, sk),
                           'jumlah': int(msk.sum()), 'terdeteksi': int(pred[msk].sum())})
    for lab in pd.unique(lab_arr):
        msk = lab_arr == lab
        rows_l.append({'rancangan': nama, 'label': lab, 'jumlah': int(msk.sum()), 'ditandai_anomali': int(pred[msk].sum())})
per_s = pd.DataFrame(rows_s); per_s['detection_rate'] = per_s['terdeteksi'] / per_s['jumlah']
per_l = pd.DataFrame(rows_l); per_l['persen_ditandai'] = per_l['ditandai_anomali'] / per_l['jumlah']
per_s.to_csv(os.path.join(HASIL, 'tabel_per_skenario.csv'), index=False)
per_l.sort_values(['label', 'rancangan']).to_csv(os.path.join(HASIL, 'tabel_per_label.csv'), index=False)
tab_s = per_s.pivot(index='skenario', columns='rancangan', values='detection_rate')[list(rancangan)]
(tab_s * 100).round(2)

In [ ]:
INK, INK2, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
WARNA = {'A': '#2a78d6', 'B': '#eb6834', 'A+C': '#1baf7a'}
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
                     'axes.edgecolor': AXIS, 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': GRID, 'grid.linewidth': 0.6,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False, 'font.size': 9})
urut = [s for s in ('HTTP flood / DoS', 'Port scanning', 'SSH brute force') if s in tab_s.index]
x, lebar = np.arange(len(urut)), 0.26
fig, ax = plt.subplots(figsize=(8, 4.4))
for i, n in enumerate(rancangan):
    v = tab_s.loc[urut, n].to_numpy() * 100
    pos = x + (i - 1) * (lebar + 0.02)
    ax.bar(pos, v, width=lebar, color=WARNA[n], label=judul[n], zorder=2)
    for xx, vv in zip(pos, v):
        ax.annotate(('%.1f' % vv).replace('.', ','), (xx, vv), xytext=(0, 3), textcoords='offset points',
                    ha='center', fontsize=8, color=INK2)
ax.set_xticks(x, urut)
ax.set_ylim(0, 112)
ax.set_ylabel('Detection rate (%)')
ax.set_title('Detection rate per skenario pada data uji CIC-IDS2017 (FPR validasi \u2264 %d%%)' % (BATAS_FPR * 100),
             loc='left', fontsize=11, fontweight='bold', color=INK, pad=34)
ax.legend(loc='upper left', ncol=1, bbox_to_anchor=(0, 1.12), fontsize=8)
ax.spines['left'].set_visible(False)
fig.savefig(os.path.join(HASIL, 'gambar_perbandingan_per_skenario.png'), dpi=200, bbox_inches='tight')
plt.show()

## Analisis: kenapa B gagal dan A+C berhasil?

Isolation Forest memilih **fitur secara acak** di setiap cabang pohon. Pada B, hanya 7 dari 75 fitur
yang berupa agregasi, sehingga keanehan seperti "1.000 port berbeda per menit" jarang dipakai untuk
memisahkan data dan **tenggelam** oleh 68 fitur aliran yang tampak normal. Satu aliran port scan
memang terlihat seperti koneksi biasa (2 paket, 0 byte).

Pada A+C, model C **hanya** melihat perilaku sumber, sehingga setiap cabangnya memakai fitur agregasi.

Sel berikut memeriksa kontribusi tiap model dan seberapa besar hasil ini bergantung pada satu IP penyerang.

In [ ]:
pA = mA.skor(test) > mA.ambang(pil_AC[wC][0]['A'])
pC = mC[wC].skor(test) > mC[wC].ambang(pil_AC[wC][0]['C'])
y = y_test == 1
ip172 = test['Source IP'].to_numpy() == '172.16.0.1'
analisis = pd.Series({
    'Serangan terdeteksi hanya oleh A': int((pA & ~pC & y).sum()),
    'Serangan terdeteksi hanya oleh C': int((pC & ~pA & y).sum()),
    'Serangan terdeteksi oleh keduanya': int((pA & pC & y).sum()),
    'Alarm palsu dari A': int((pA & ~y).sum()),
    'Alarm palsu dari C': int((pC & ~y).sum()),
    'Aliran serangan dari 172.16.0.1': int((ip172 & y).sum()),
    'Aliran serangan dari IP lain': int((~ip172 & y).sum()),
})
analisis.to_frame('jumlah')

**Catatan untuk pembahasan:**
1. Hampir semua serangan CIC-IDS2017 berasal dari satu IP (`172.16.0.1`), sehingga model perilaku sumber (C)
   sangat terbantu. Di internet, penyerang dapat memakai banyak IP, karena itu agregasi per subnet /24 tetap
   disertakan dan sistem tetap diuji pada server sendiri.
2. Pada rancangan terpilih, hampir semua deteksi berasal dari model C. Model A tetap dipertahankan untuk
   anomali yang tampak dari satu koneksi saja.
3. Waktu CIC-IDS2017 hanya sampai menit, sehingga jendela agregasinya bersifat per menit.

## Simpan model & ringkasan

In [ ]:
cs = pil_AC[wC][0]
joblib.dump({'A': {'model': mA.model, 'scaler': mA.scaler, 'fitur': mA.fitur,
                   'contamination': cs['A'], 'threshold': mA.ambang(cs['A'])},
             'C': {'model': mC[wC].model, 'scaler': mC[wC].scaler, 'fitur': mC[wC].fitur,
                   'contamination': cs['C'], 'threshold': mC[wC].ambang(cs['C'])},
             'jendela_detik': wC}, os.path.join(HASIL, 'model_A+C.joblib'))
ringkas = {'dataset': 'CIC-IDS2017 (GeneratedLabelledFlows), preprocessing v2',
           'parameter': {k: v for k, v in PARAM.items() if k != 'n_jobs'},
           'aturan_pemilihan': 'F1 validasi tertinggi dengan FPR validasi <= %.2f' % BATAS_FPR,
           'data': {'latih': len(train), 'validasi': len(val), 'uji': len(test)},
           'jendela_terpilih_detik': {'B': wB, 'A+C': wC},
           'hasil_uji': hasil,
           'analisis_A+C': {k: int(v) for k, v in analisis.items()},
           'detection_rate_per_skenario': {n: {k: round(float(v), 4) for k, v in tab_s[n].items()} for n in rancangan}}
json.dump(ringkas, open(os.path.join(HASIL, 'ringkasan_hasil_v2.json'), 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print('Tersimpan di', HASIL)
for f in sorted(os.listdir(HASIL)):
    print('  ', f)